<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-10-production-deployment/lesson-10.2-designing-reliability/notebooks/exercises-10_2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 10.2: Designing for Reliability

*Module 10 · AI System Architecture & Production Deployment*

> LLM APIs go down. Networks flake. Rate limits hit. Tokens run out. A reliable AI system doesn't prevent failures — it survives them gracefully. This lesson builds the four reliability patterns every production system needs: circuit breakers, retries, fallbacks, and graceful degradation.

---

## About this notebook

This notebook contains the **5 runnable code examples** from the published lesson `lesson-10.2.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Circuit Breaker — Stop Hitting a Dead Service — `part1_circuit_breaker.py`
2. Step 2 — Retry Engine — Exponential Backoff + Jitter — `part2_retry.py`
3. Step 3 — Fallback Provider Manager — Automatic Failover with Health Tracking — `part3_fallback.py`
4. Step 4 — Graceful Degradation — Something Is Always Better Than Nothing — `part4_degradation.py`
5. Step 5 — ResilientAIClient — All Patterns in One Class — `part5_resilient_client.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai requests


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Circuit Breaker — Stop Hitting a Dead Service

After N failures, stop calling. Wait. Test with one request. If it works, resume.

**`part1_circuit_breaker.py`** · _Block 1 of 5_

CIRCUIT BREAKER — 3-state machine: CLOSED → OPEN → HALF_OPEN


In [ ]:
# =============================================
# CIRCUIT BREAKER
# 3-state machine: CLOSED → OPEN → HALF_OPEN
# Prevents cascading failures by fast-failing
# when a service is known to be down.
# =============================================

import time
from enum import Enum
from dataclasses import dataclass, field
from typing import Callable, Any

class State(Enum):
    CLOSED = "closed"        # normal: requests flow through
    OPEN = "open"            # broken: requests rejected instantly
    HALF_OPEN = "half_open"  # testing: allow ONE request to check recovery

class CircuitBreaker:
    """Protect against cascading failures from a broken service."""
    
    def __init__(self, name: str,
                 failure_threshold: int = 5,
                 recovery_timeout: int = 30,
                 half_open_max: int = 1):
        self.name = name
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.half_open_max = half_open_max
        
        self.state = State.CLOSED
        self.failure_count = 0
        self.success_count = 0
        self.last_failure_time = 0.0
        self.half_open_calls = 0
        
        # Metrics
        self.total_calls = 0
        self.total_blocked = 0
        self.total_successes = 0
        self.total_failures = 0
    
    def call(self, fn: Callable, *args, **kwargs) -> Any:
        """Execute fn through the circuit breaker."""
        self.total_calls += 1
        
        # OPEN: reject immediately (fast fail)
        if self.state == State.OPEN:
            if time.time() - self.last_failure_time > self.recovery_timeout:
                self.state = State.HALF_OPEN
                self.half_open_calls = 0
            else:
                self.total_blocked += 1
                raise CircuitOpenError(
                    f"Circuit '{self.name}' is OPEN. "
                    f"Retry in {self.recovery_timeout - int(time.time() - self.last_failure_time)}s"
                )
        
        # HALF_OPEN: allow limited test requests
        if self.state == State.HALF_OPEN:
            if self.half_open_calls >= self.half_open_max:
                self.total_blocked += 1
                raise CircuitOpenError(f"Circuit '{self.name}' is HALF_OPEN, test in progress")
            self.half_open_calls += 1
        
        # Execute
        try:
            result = fn(*args, **kwargs)
            self._on_success()
            return result
        except Exception as e:
            self._on_failure()
            raise
    
    def _on_success(self):
        self.total_successes += 1
        if self.state == State.HALF_OPEN:
            self.state = State.CLOSED  # recovered!
            self.failure_count = 0
        elif self.state == State.CLOSED:
            self.failure_count = max(0, self.failure_count - 1)  # heal slowly
    
    def _on_failure(self):
        self.total_failures += 1
        self.failure_count += 1
        self.last_failure_time = time.time()
        
        if self.state == State.HALF_OPEN:
            self.state = State.OPEN    # test failed, reopen
        elif self.failure_count >= self.failure_threshold:
            self.state = State.OPEN    # too many failures, trip
    
    @property
    def stats(self) -> dict:
        return {"name": self.name, "state": self.state.value,
                "failures": self.failure_count, "threshold": self.failure_threshold,
                "total_calls": self.total_calls, "blocked": self.total_blocked}

class CircuitOpenError(Exception):
    pass

# ── Demo ──
cb = CircuitBreaker("gemini_api", failure_threshold=3, recovery_timeout=5)
call_num = 0

def flaky_api(prompt):
    """Fails for calls 3-7, then recovers."""
    global call_num; call_num += 1
    if 3 <= call_num <= 7: raise ConnectionError("API timeout")
    return f"Response to: {prompt}"

print("Circuit breaker demo:\n")
for i in range(12):
    try:
        result = cb.call(flaky_api, f"query-{i}")
        print(f"  Call {i+1:2d}: ✅ {cb.state.value:10s} → {result[:30]}")
    except CircuitOpenError as e:
        print(f"  Call {i+1:2d}: 🚫 {cb.state.value:10s} → BLOCKED (fast fail)")
    except Exception as e:
        print(f"  Call {i+1:2d}: ❌ {cb.state.value:10s} → {e}")
    time.sleep(1)

print(f"\nStats: {cb.stats}")


> **What just happened?** A 3-state circuit breaker: CLOSED (normal — requests flow through). After 3 failures → OPEN (broken — requests rejected instantly with CircuitOpenError , no network call). After 5 seconds → HALF_OPEN (testing — allow 1 test request). If test succeeds → CLOSED. If test fails → OPEN again. The demo shows: calls 1-2 succeed, 3-5 fail (counts up), call 6 trips the circuit, calls 7-10 are blocked instantly (fast fail, zero latency), then recovery. Without the circuit breaker, calls 6-10 would each wait 30 seconds for a timeout. With it, they fail in microseconds.


### Step 2 · Retry Engine — Exponential Backoff + Jitter

Transient errors (429, 503) often resolve on their own. Retry — but smarter each time.

**`part2_retry.py`** · _Block 2 of 5_

RETRY ENGINE — Exponential backoff with jitter.


In [ ]:
# =============================================
# RETRY ENGINE
# Exponential backoff with jitter.
# Respects Retry-After headers.
# Classifies errors as retryable vs permanent.
# =============================================

import time, random
from dataclasses import dataclass

@dataclass
class RetryConfig:
    max_retries: int = 3
    base_delay: float = 1.0       # initial wait (seconds)
    max_delay: float = 30.0       # cap the wait time
    exponential_base: float = 2.0 # multiplier per retry
    jitter: bool = True           # add randomness to prevent thundering herd

# Errors worth retrying vs permanent failures
RETRYABLE_ERRORS = (
    ConnectionError, TimeoutError, OSError,
)

RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}

class RetryEngine:
    """Retry with exponential backoff, jitter, and error classification."""
    
    def __init__(self, config: RetryConfig = None):
        self.config = config or RetryConfig()
        self.attempt_log: list[dict] = []
    
    def _is_retryable(self, error: Exception) -> bool:
        """Should we retry this error?"""
        if isinstance(error, RETRYABLE_ERRORS):
            return True
        # Check for HTTP status code in error message
        err_str = str(error)
        for code in RETRYABLE_STATUS_CODES:
            if str(code) in err_str:
                return True
        return False
    
    def _calculate_delay(self, attempt: int, error: Exception = None) -> float:
        """Calculate wait time with exponential backoff + jitter."""
        # Check for Retry-After header in error
        err_str = str(error) if error else ""
        if "retry-after" in err_str.lower():
            try:
                import re
                match = re.search(r"retry.after.*?(\d+)", err_str, re.IGNORECASE)
                if match:
                    return float(match.group(1))
            except:
                pass
        
        # Exponential backoff
        delay = min(
            self.config.base_delay * (self.config.exponential_base ** attempt),
            self.config.max_delay,
        )
        
        # Jitter: randomize ±30% to prevent thundering herd
        if self.config.jitter:
            delay *= random.uniform(0.7, 1.3)
        
        return round(delay, 2)
    
    def execute(self, fn, *args, **kwargs) -> dict:
        """Execute with retry logic."""
        last_error = None
        
        for attempt in range(self.config.max_retries + 1):
            try:
                start = time.time()
                result = fn(*args, **kwargs)
                elapsed = round(time.time() - start, 3)
                
                self.attempt_log.append({
                    "attempt": attempt + 1, "success": True, "elapsed_s": elapsed,
                })
                return {"result": result, "attempts": attempt + 1, "success": True}
            
            except Exception as e:
                last_error = e
                
                # Don't retry non-retryable errors
                if not self._is_retryable(e):
                    self.attempt_log.append({"attempt": attempt + 1, "error": str(e), "retryable": False})
                    return {"error": str(e), "attempts": attempt + 1, "success": False, "retryable": False}
                
                # Last attempt? Don't wait.
                if attempt == self.config.max_retries:
                    break
                
                delay = self._calculate_delay(attempt, e)
                self.attempt_log.append({"attempt": attempt + 1, "error": str(e), "delay_s": delay})
                
                print(f"    Retry {attempt+1}/{self.config.max_retries}: waiting {delay}s...")
                time.sleep(delay)
        
        return {"error": str(last_error), "attempts": self.config.max_retries + 1, "success": False}

# ── Demo ──
retry = RetryEngine(RetryConfig(max_retries=3, base_delay=0.5))
attempt = 0

def fails_twice(prompt):
    global attempt; attempt += 1
    if attempt <= 2: raise ConnectionError("503 Service Unavailable")
    return f"Success on attempt {attempt}!"

print("Retry demo (fails twice, succeeds on 3rd):\n")
result = retry.execute(fails_twice, "test query")
print(f"\n  Result: {result}")

print("""
  Delay schedule (base=1s, exponential_base=2):
    Attempt 1 fails → wait 1.0s  (± 30% jitter)
    Attempt 2 fails → wait 2.0s  (± 30% jitter)
    Attempt 3 fails → wait 4.0s  (± 30% jitter)
    Attempt 4 fails → give up

  Jitter prevents thundering herd:
    Without jitter: 1000 clients all retry at exactly 1s, 2s, 4s
    With jitter:    clients spread across 0.7-1.3s, 1.4-2.6s, 2.8-5.2s
""")


> **What just happened?** RetryEngine with three smart features: (1) Error classification — ConnectionError, 429, 503 are retryable; 400 (bad request) or 401 (auth) are permanent (don't waste retries). (2) Exponential backoff — 1s → 2s → 4s → 8s (capped at 30s). Each retry waits longer. (3) Jitter — ±30% randomness prevents thundering herd (1000 clients retrying at exactly the same second). Also respects Retry-After headers from the API. The demo fails twice, succeeds on attempt 3.


### Step 3 · Fallback Provider Manager — Automatic Failover with Health Tracking

Three providers. If one is down, seamlessly switch to the next. Track health over time.

**`part3_fallback.py`** · _Block 3 of 5_

FALLBACK PROVIDER MANAGER — Multi-provider failover with per-provider


In [ ]:
# =============================================
# FALLBACK PROVIDER MANAGER
# Multi-provider failover with per-provider
# circuit breakers and health scoring.
# =============================================

from dataclasses import dataclass, field

@dataclass
class ProviderConfig:
    name: str
    fn: object                   # callable(prompt) → str
    priority: int = 0           # lower = preferred
    cost_per_1k: float = 0.01   # cost per 1K tokens

class FallbackManager:
    """Manage multiple LLM providers with automatic failover."""
    
    def __init__(self, providers: list[ProviderConfig]):
        self.providers = sorted(providers, key=lambda p: p.priority)
        self.breakers = {p.name: CircuitBreaker(p.name, failure_threshold=3, recovery_timeout=30)
                         for p in providers}
        self.retry = RetryEngine(RetryConfig(max_retries=2, base_delay=0.5))
        self.call_log: list[dict] = []
    
    def call(self, prompt: str) -> dict:
        """Try providers in priority order with retry + circuit breaker."""
        errors = []
        
        for provider in self.providers:
            breaker = self.breakers[provider.name]
            
            # Skip if circuit breaker is open
            if breaker.state == State.OPEN:
                if time.time() - breaker.last_failure_time <= breaker.recovery_timeout:
                    errors.append({"provider": provider.name, "error": "circuit open"})
                    continue
            
            # Try with retry
            def guarded_call(p=prompt):
                return breaker.call(provider.fn, p)
            
            result = self.retry.execute(guarded_call)
            
            if result["success"]:
                entry = {"provider": provider.name, "attempts": result["attempts"],
                         "fallbacks_skipped": len(errors)}
                self.call_log.append(entry)
                return {"text": result["result"], "provider": provider.name, **entry}
            
            errors.append({"provider": provider.name, "error": result.get("error", "unknown")})
        
        # All providers failed
        return {"text": None, "provider": "none", "errors": errors, "all_failed": True}
    
    def health_report(self):
        print("\n  Provider Health:")
        for p in self.providers:
            b = self.breakers[p.name]
            icon = {State.CLOSED: "🟢", State.OPEN: "🔴", State.HALF_OPEN: "🟡"}[b.state]
            print(f"    {icon} {p.name:15s} state={b.state.value:10s} fails={b.failure_count} blocked={b.total_blocked}")

# ── Demo ──
import google.generativeai as genai
import os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY", ""))

def gemini_primary(prompt):
    raise ConnectionError("Primary Gemini down")  # simulate failure

def gemini_backup(prompt):
    return f"[Backup Gemini] Response for: {prompt[:30]}"

def local_fallback(prompt):
    return "[Local] I'm operating with limited capability. Please try again later."

manager = FallbackManager([
    ProviderConfig("gemini_primary", gemini_primary, priority=0, cost_per_1k=0.01),
    ProviderConfig("gemini_backup",  gemini_backup,  priority=1, cost_per_1k=0.01),
    ProviderConfig("local_fallback", local_fallback,  priority=2, cost_per_1k=0.0),
])

print("Fallback demo:\n")
result = manager.call("What is MCP?")
print(f"  Result: {result['text'][:60]}")
print(f"  Provider: {result['provider']} (skipped {result.get('fallbacks_skipped', 0)} failed providers)")
manager.health_report()


> **What just happened?** FallbackManager combines retry + circuit breaker + provider priority. Each provider gets its own circuit breaker. For every request: try the primary with 2 retries → if it fails, the circuit breaker opens → seamlessly try the backup → if that fails too → try the local fallback (zero-cost, limited capability). The health report shows each provider's status. The key insight: each layer of reliability compounds. Retry handles transient errors. Circuit breaker handles persistent failures. Fallback handles complete outages.


### Step 4 · Graceful Degradation — Something Is Always Better Than Nothing

When all providers fail: serve cached responses, simpler models, or honest "limited mode" messages.

**`part4_degradation.py`** · _Block 4 of 5_

GRACEFUL DEGRADATION — When everything fails, degrade — don't crash.


In [ ]:
# =============================================
# GRACEFUL DEGRADATION
# When everything fails, degrade — don't crash.
# 4 degradation levels from full to minimal.
# =============================================

import hashlib, json

class DegradationLevel(Enum):
    FULL = "full"               # normal operation
    REDUCED = "reduced"         # cheaper model, shorter responses
    CACHED = "cached"           # serve from cache only
    MINIMAL = "minimal"         # static fallback messages

class GracefulDegrader:
    """Progressively degrade service quality instead of failing completely."""
    
    def __init__(self):
        self.level = DegradationLevel.FULL
        self.cache: dict[str, dict] = {}
        self.static_responses = {
            "greeting": "Hello! I'm currently operating with limited capability. I can answer basic questions.",
            "error": "I'm having trouble processing your request. Please try again in a few minutes.",
            "courses": "We offer GenAI (₹29,999), ML Engineering (₹39,999), and Python Foundations (₹9,999). Visit netsetos.com for details.",
        }
    
    def cache_response(self, prompt: str, response: str):
        """Cache a successful response for degraded mode."""
        key = hashlib.md5(prompt.lower().strip().encode()).hexdigest()
        self.cache[key] = {"response": response, "time": time.time()}
    
    def get_cached(self, prompt: str, max_age: int = 3600) -> str:
        """Get a cached response if available and fresh."""
        key = hashlib.md5(prompt.lower().strip().encode()).hexdigest()
        cached = self.cache.get(key)
        if cached and time.time() - cached["time"] < max_age:
            return cached["response"]
        return None
    
    def get_static(self, prompt: str) -> str:
        """Get the best static response for a prompt."""
        prompt_lower = prompt.lower()
        if any(w in prompt_lower for w in ["hi", "hello", "hey"]):
            return self.static_responses["greeting"]
        if any(w in prompt_lower for w in ["course", "price", "cost", "enroll"]):
            return self.static_responses["courses"]
        return self.static_responses["error"]
    
    def respond(self, prompt: str, llm_fn = None,
                cheap_fn = None) -> dict:
        """Respond at the current degradation level."""
        
        # Level 1: FULL — use the best model
        if self.level == DegradationLevel.FULL and llm_fn:
            try:
                response = llm_fn(prompt)
                self.cache_response(prompt, response)  # cache for later
                return {"text": response, "level": "full"}
            except:
                self.level = DegradationLevel.REDUCED   # auto-degrade
        
        # Level 2: REDUCED — use a cheaper/faster model
        if self.level == DegradationLevel.REDUCED and cheap_fn:
            try:
                response = cheap_fn(prompt)
                self.cache_response(prompt, response)
                return {"text": response, "level": "reduced",
                        "notice": "Responses may be shorter than usual."}
            except:
                self.level = DegradationLevel.CACHED
        
        # Level 3: CACHED — serve from cache
        if self.level == DegradationLevel.CACHED:
            cached = self.get_cached(prompt)
            if cached:
                return {"text": cached, "level": "cached",
                        "notice": "This is a cached response and may be outdated."}
            self.level = DegradationLevel.MINIMAL
        
        # Level 4: MINIMAL — static responses
        return {"text": self.get_static(prompt), "level": "minimal",
                "notice": "I'm currently in limited mode. Full service will resume shortly."}

# ── Demo ──
degrader = GracefulDegrader()

# Pre-populate cache
degrader.cache_response("What is MCP?", "MCP (Model Context Protocol) is an open standard by Anthropic for connecting AI models to tools.")

print("Degradation levels:\n")
for level in DegradationLevel:
    degrader.level = level
    r = degrader.respond("What is MCP?")
    print(f"  {level.value:8s} → {r['text'][:70]}...")
    if r.get("notice"): print(f"           ℹ️ {r['notice']}")


> **What just happened?** GracefulDegrader with 4 levels: FULL (best model, cache the response). REDUCED (cheaper model, shorter responses, add a notice). CACHED (serve from cache, warn about staleness). MINIMAL (static keyword-matched responses — "We offer GenAI at ₹29,999..."). The system auto-degrades: if the primary model fails → drops to REDUCED → if that fails too → CACHED → MINIMAL. The user always gets something. A "limited mode" response is infinitely better than an error page.


### Step 5 · ResilientAIClient — All Patterns in One Class

Circuit breaker + retry + fallback + degradation = production-grade reliability.

**`part5_resilient_client.py`** · _Block 5 of 5_

RESILIENT AI CLIENT — All reliability patterns unified.


In [ ]:
# =============================================
# RESILIENT AI CLIENT
# All reliability patterns unified.
# Circuit breaker → Retry → Fallback → Degrade.
# =============================================

class ResilientAIClient:
    """Production-ready AI client with full reliability stack."""
    
    def __init__(self, providers: list[ProviderConfig]):
        self.fallback = FallbackManager(providers)
        self.degrader = GracefulDegrader()
        self.metrics = {"total": 0, "success": 0, "fallback": 0, "degraded": 0, "failed": 0}
    
    def chat(self, prompt: str) -> dict:
        """Send a prompt with full reliability protection."""
        self.metrics["total"] += 1
        
        # Try through the fallback chain (retry + circuit breaker per provider)
        result = self.fallback.call(prompt)
        
        if result.get("text"):
            # Cache successful response for degradation
            self.degrader.cache_response(prompt, result["text"])
            
            if result.get("fallbacks_skipped", 0) > 0:
                self.metrics["fallback"] += 1
            else:
                self.metrics["success"] += 1
            
            return {"text": result["text"], "provider": result["provider"], "degraded": False}
        
        # All providers failed → graceful degradation
        self.metrics["degraded"] += 1
        self.degrader.level = DegradationLevel.CACHED
        degraded = self.degrader.respond(prompt)
        
        return {"text": degraded["text"], "provider": "degraded",
                "degraded": True, "level": degraded["level"],
                "notice": degraded.get("notice", "")}
    
    def report(self):
        m = self.metrics
        total = m["total"] or 1
        print(f"\n  📊 Reliability Report")
        print(f"  {'─'*40}")
        print(f"  Total requests:   {m['total']}")
        print(f"  Primary success:  {m['success']} ({m['success']/total:.0%})")
        print(f"  Fallback used:    {m['fallback']} ({m['fallback']/total:.0%})")
        print(f"  Degraded:         {m['degraded']} ({m['degraded']/total:.0%})")
        print(f"  User-facing err:  {m['failed']} ({m['failed']/total:.0%})")
        print(f"  Effective uptime: {(total - m['failed'])/total:.2%}")
        self.fallback.health_report()

# ── Run ──
client = ResilientAIClient([
    ProviderConfig("gemini_primary", gemini_primary, priority=0),
    ProviderConfig("gemini_backup",  gemini_backup,  priority=1),
    ProviderConfig("local_fallback", local_fallback,  priority=2),
])

# Pre-cache some responses
client.degrader.cache_response("What courses do you offer?", "We offer GenAI Complete (₹29,999), ML Engineering (₹39,999), and Python Foundations (₹9,999).")

print("ResilientAIClient demo:\n")
for prompt in ["What is MCP?", "Tell me about RAG", "What courses do you offer?"]:
    result = client.chat(prompt)
    icon = "✅" if not result["degraded"] else "⚠️"
    print(f"  {icon} \"{prompt[:30]}...\" → {result['provider']} | {result['text'][:50]}...")

client.report()


> **What just happened?** ResilientAIClient integrates every pattern into one call: (1) Try primary provider with retry + circuit breaker. (2) If primary fails, try backup (also with retry + circuit breaker). (3) If backup fails, try local fallback. (4) If everything fails, serve cached or static response via graceful degradation. Every successful response is cached for future degradation. The reliability report shows: primary success rate, fallback rate, degradation rate, and effective uptime. User-facing errors should be 0% — the degradation layer always returns something.


---

## Wrap-up

You've walked through all 5 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
